# BNB token report

One BEP-20 token on opBNB or BNB Smart Chain: how far finality trails the chain, the token's holders over recent blocks, and whether its largest holders have revealed their public keys. The defaults read USDT on opBNB mainnet and need no key; on BNB Smart Chain set `NODEREAL_API_KEY` (free) in `.env`. Nothing is stored.

In [ ]:
import pandas as pd

from multichain_indexer import Indexer
from multichain_indexer.core.config import Settings

CHAIN, NETWORK = "opbnb", "mainnet"
# Tether USD on opBNB mainnet.
BEP20 = "0x9e5aac1ba1a2e6aed6b32689dfcf62a509ca96f3"
BLOCKS = 1_200

indexer = Indexer(CHAIN, NETWORK, settings=Settings.from_env())

## Finality

In [ ]:
finality = indexer.finality()
pd.Series(finality.model_dump())

## Holders over the newest blocks

Only addresses that moved the token in the range are counted, so the top-ten share is a lower bound.

In [ ]:
holders = indexer.token_holders(BEP20, from_block=finality.head_block - BLOCKS + 1)
summary = holders.model_dump(exclude={"top_holders", "transfers_per_day"})
pd.Series(summary)

In [ ]:
top = pd.DataFrame([holder.model_dump() for holder in holders.top_holders])
top

In [ ]:
pd.Series(holders.transfers_per_day, name="transfers")

## Public keys of the largest holders

An account with a nonce above zero has signed a transaction, so its public key is on chain. The rule is a lower bound: a permit or other EIP-712 signature used by someone else reveals a key while the nonce is still zero.

In [ ]:
exposure = indexer.key_exposure(list(top["address"])) if len(top) else None
pd.DataFrame([row.model_dump() for row in exposure.accounts]) if exposure else None

In [ ]:
indexer.close()